# Transformer 流程

> **一句话**：Transformer 是"把 RNN 换成注意力"的 Seq2Seq 翻译模型——**编码器**把源句读成一组向量（记忆），**解码器**一边看这组向量、一边把译文逐个写出来。
>
> 本笔记按数据流的顺序讲完整流程；对照代码为 week19 的 `attention-is-all-you-need-pytorch`，手写实现见 `0925.ipynb`。

## 总流程一览

```text
[源句 "我 爱 你"]
      │ ① 词嵌入（查表）   ② + 位置编码（告诉模型词序）
      ▼
[Encoder × N 层：多头自注意力 → 残差+LN → FFN → 残差+LN]
      │ 输出 = "记忆" memory（对源句的理解，每个词一个向量）
      ▼
[Decoder × N 层：掩码自注意力 → 交叉注意力 → FFN（各带残差+LN）]
      │ 输出 = 译文中每个位置的向量
      ▼
[线性层 + softmax] → 每个位置一个"下一个词"的概率分布
```

- **训练时**：目标句整句一次性并行输入，每个位置同时预测下一个词（靠掩码不许偷看未来）。
- **推理时**：从 `<s>` 开始逐个生成，生成一个词就拼回去当新的输入（自回归）。

## 第 ① ② 步｜输入：词嵌入 + 位置编码

- **词嵌入**：把词 id 查表变成 `d_model` 维向量（一张可学习的查找表）。论文 $d_{model}=512$，我们骨架用 128。
- **为什么需要位置编码**：注意力本质是"加权求和"，对词序不敏感——把"我爱你"打乱输入，输出只是跟着换位置，模型分不清"我爱你"和"你爱我"。
- **做法**：给每个位置准备一个固定的正弦/余弦向量，直接**加到**词向量上：

$$PE_{(pos,\,2i)}=\sin\!\Big(\frac{pos}{10000^{2i/d_{model}}}\Big),\qquad
PE_{(pos,\,2i+1)}=\cos\!\Big(\frac{pos}{10000^{2i/d_{model}}}\Big)$$

- 直觉：不同维度是不同频率的"刻度盘"（低频=时针、高频=秒针），合起来唯一定位一个位置；点积还能表达相对距离。
- 📌 代码里（`Models.py` → `PositionalEncoding`）：预先算好整张表、`register_buffer` 存着（不参与训练），前向时切片相加；仓库在这里还额外过了一层 LayerNorm（论文没有，属实现细节）。

## 第 ③ 步｜注意力：通俗版"查资料"

把注意力想成一次**检索**，每个词都有三个角色：

| 角色 | 含义 | 类比 |
| --- | --- | --- |
| Query (Q) | 我现在想找什么 | 搜索框里输入的关键字 |
| Key (K) | 我能提供什么 | 每篇文章的标题/标签 |
| Value (V) | 我真正提供的内容 | 文章正文 |

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

四步走：**①打分**（当前词与每个词的相关度）→ **②缩放** $1/\sqrt{d_k}$（防止分数过大、softmax 过尖）→ **③归一化** softmax（和为 1 的权重）→ **④加权求和** V（按权重提取信息）。

- **多头注意力**：把 `d_model` 切成 h 份，各自做一套 Q/K/V（各学一种关系：主谓、指代、修饰……），拼回后过一个线性层。就像 h 个读者从不同角度同时读同一句话。
- 与 0921 的联系：加性/乘性注意力本质上都是"打分 → softmax → 加权"，Transformer 只是用**缩放点积**并一次性矩阵并行算完。
- 遮蔽（mask）：把 `<pad>` 或"未来的词"的分数填成 $-\infty$，softmax 后这些位置权重为 0。（0925 的 M3 用的就是 `float('-inf')`。）

## 第 ④ 步｜编码器：把源句"读透"

编码器 = 同一个 `EncoderLayer` 叠 N 层（论文 6 层），每层就两个子层：

1. **多头自注意力**（Q=K=V 都来自源句）：每个词都能看到整句的每个词，长距离依赖一步到位（不像 RNN 要一步步传）。
2. **FFN 前馈网络**：两层线性 + ReLU，逐位置独立做非线性加工（$d_{model}\to d_{ff}\to d_{model}$，论文 512→2048→512）。

两个子层都包在**残差 + LayerNorm** 里：`x = LayerNorm(x + 子层(x))`。

- 残差：保底（最差学成"什么都没变"）+ 梯度直通，是能堆深的原因。
- LayerNorm：把每个位置的向量拉回稳定的数值范围。
- 维度必须保持 `d_model`，否则没法相加——这是 Transformer 所有模块的"公共货币"。

📌 浅层学局部语法、深层学抽象语义；编码器最后输出的 `memory`（每个源词一个向量）会被解码器每一层使用。

## 第 ⑤⑥⑦ 步｜解码器：边看源句边写译文

解码器每层有三个子层，比编码器多出两种注意力：

**⑤ 掩码自注意力（看译文自己，但不许偷看未来）**

- 训练时译文整句并行输入，如果第 1 个位置能看见第 2 个词，就是"开卷考试"，推理时根本学不会生成。
- 所以把"未来"位置的分数填 $-\infty$（下三角掩码）：`get_subsequent_mask` 造下三角，再和 `<pad>` 掩码 `get_pad_mask` 相与 → `trg_mask`。

**⑥ 交叉注意力（连接源句与译文）——就是 0921 学过的那个注意力**

- **Q 来自解码器**（"我正生成这个词，想找对应的源句信息"），**K、V 来自编码器 memory**（源句的全部理解）。
- 生成 "Tom" 时会去源句"瞟一眼"「汤姆」——注意力热力图就是画在这里的。

**⑦ FFN**：同编码器，逐位置 MLP。

每层结尾同样是**残差 + LayerNorm**；解码器输出 `[B, Ltgt, d_model]`，进入下一步的输出层。

## 第 ⑧ 步｜输出层：从向量到"下一个词"

- **线性层**：`[B, Ltgt, d_model] → [B, Ltgt, V]`，得到词表里每个词的原始分数（logits）。
- **softmax**：把分数变成概率分布；取最大者为贪心输出，或交给 beam search 保留 top-k 候选。
- 两个实现细节（对应仓库）：
  - **权重共享**：输出线性层与（目标词）嵌入共享同一个权重矩阵，再乘 $1/\sqrt{d_{model}}$ 平衡尺度（`scale_emb_or_prj='prj'` 的默认行为）。
  - **训练目标**：每个位置与真实"下一个词"算交叉熵；仓库加了 label smoothing（0.1），防止模型过度自信。

## 训练 vs 推理：两种模式别混淆

**训练（teacher forcing，整句并行）**

```text
"<s> I love you"  ──一次前向──►  4 个位置同时预测 "I love you </s>"
掩码保证位置 t 只能看到 ≤ t 的词 → 一次算完所有位置的交叉熵 → 反传
```

**推理（自回归，逐个生成）**

```text
<s> ──► I ──► love ──► you ──► </s>    每步只产一个词，拼回输入端再喂
beam search：每步保留 k 条候选（beam），最后按 得分/长度^0.7 选整句
```

- 训练并行 ⇒ 必须用掩码防偷看；推理串行 ⇒ 天生看不到未来，不需要掩码，但慢。
- 对比 0921 的 GRU 版：它只有"整句喂入算 loss"这一半并行（teacher forcing），编码仍是串行读；Transformer 把**编码与训练全流程都并行化**，这是最能堆深、最能吃算力红利的原因。

## 每个设计"为什么" + 一句话总结

| 设计 | 为什么 |
| --- | --- |
| 缩放 $1/\sqrt{d_k}$ | $d_k$ 越大点积方差越大，softmax 过尖 → 梯度消失 |
| 残差连接 | 梯度直通 + 保底不退化，能堆几十层 |
| LayerNorm（而非 BN） | 逐样本、逐位置归一化，天然适配变长序列 |
| 位置编码"相加"而非拼接 | 不增维度，模型自己学"语义+位置"的组合；正弦表可外推更长序列 |
| 解码器用两种注意力 | 自注意力管"译文内部连贯"，交叉注意力管"对准源句" |
| 训练要 mask | 整句并行时必须挡住未来词，否则学不到"生成" |

**背**：查表得词向量 → **+位置编码** → **编码器 N 层**（自注意力 + FFN，各带残差/LN）把源句读成 memory → **解码器 N 层**（掩码自注意力 + 交叉注意力 + FFN）边看 memory 边生成 → **线性 + softmax** 出下一个词。